# CHALLENGE 2026 – Data Science and Statistical Computing
## Aura AI – Inteligência Conversacional

**Integrantes:** João Guilherme (RM565244), Matheus Kitamura (RM563205), Gustavo Barroso (RM565705), Victor Alves (RM565723)

---

## Objetivo

A Aura AI é uma solução de Inteligência Artificial desenvolvida para analisar transcrições de reuniões comerciais e transformá-las em insights estratégicos para a equipe comercial da TOTVS.

Este notebook apresenta o processo de preparação dos dados, aplicação de NLP com TF‑IDF, análise exploratória e geração de insights de negócio a partir das transcrições disponibilizadas pela empresa.


# 1. Entendimento do Problema

As reuniões comerciais geram grande volume de informações não estruturadas. Muitas dessas informações podem indicar oportunidades de venda, riscos de churn, insatisfações ou interesses em produtos específicos.

O objetivo da Aura AI é transformar essas conversas em informações estruturadas capazes de apoiar vendedores, consultores e gestores na tomada de decisão.

### Aplicações no Negócio
- Identificação de oportunidades comerciais.
- Detecção de sinais de insatisfação.
- Apoio à retenção de clientes.
- Priorização de ações comerciais.
- Geração de inteligência acionável.


In [ ]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt

from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer

df = pd.read_csv('../data/DADOS.csv')

print('Quantidade de linhas:', df.shape[0])
print('Quantidade de colunas:', df.shape[1])

df.head()


# 2. Ingestão e Preparação dos Dados

In [ ]:
STOPWORDS = {
'de','da','do','das','dos','a','o','as','os','e','em','um','uma',
'para','por','com','que','na','no','nas','nos','se','ao','aos',
'é','ser','foi','são','como','mais','já','também','ou','não'
}

def limpar_texto(texto):
    if pd.isna(texto):
        return ''
    texto = str(texto).lower()
    texto = re.sub(r'[^\w\s]', ' ', texto)
    palavras = texto.split()
    palavras = [p for p in palavras if p not in STOPWORDS]
    return ' '.join(palavras)

df['texto_limpo'] = df['ANON_TRANSCRICAO'].apply(limpar_texto)

df[['ANON_TRANSCRICAO','texto_limpo']].head()


### Problemas Encontrados

- Presença de pontuações e caracteres especiais.
- Diferenças de capitalização.
- Palavras muito frequentes sem valor analítico.
- Necessidade de normalização dos textos.

Após o tratamento, os textos ficaram aptos para aplicação das técnicas de NLP.


# 3. Feature Engineering com TF‑IDF

In [ ]:
vectorizer = TfidfVectorizer(
    max_features=100,
    ngram_range=(1,2),
    min_df=2
)

tfidf_matrix = vectorizer.fit_transform(df['texto_limpo'])

termos = vectorizer.get_feature_names_out()
pesos = tfidf_matrix.mean(axis=0).A1

tfidf_df = pd.DataFrame({
    'Termo': termos,
    'Peso_Medio': pesos
}).sort_values('Peso_Medio', ascending=False)

tfidf_df.head(20)


### Justificativa dos Parâmetros

- **max_features=100**: mantém os termos mais relevantes.
- **ngram_range=(1,2)**: considera palavras isoladas e expressões compostas.
- **min_df=2**: remove termos extremamente raros.

O TF‑IDF foi escolhido por sua simplicidade, interpretabilidade e capacidade de destacar termos relevantes dentro das reuniões.


# 4. Análise Exploratória dos Dados

In [ ]:
palavras = ' '.join(df['texto_limpo']).split()

freq = Counter(palavras)

freq_df = pd.DataFrame(
    freq.items(),
    columns=['Termo','Frequencia']
).sort_values('Frequencia', ascending=False)

freq_df.head(20)


In [ ]:
top20 = freq_df.head(20)

plt.figure(figsize=(12,6))
plt.bar(top20['Termo'], top20['Frequencia'])
plt.xticks(rotation=45)
plt.title('20 Termos Mais Frequentes')
plt.tight_layout()
plt.show()


### Interpretação

Os termos mais frequentes representam temas recorrentes nas conversas dos clientes. Esses padrões podem ser utilizados pela Aura AI para alimentar modelos de recomendação, alertas comerciais e identificação de riscos.


# 5. Introdução a Embeddings

## TF‑IDF
Representa documentos utilizando frequência e importância estatística das palavras.

## Embeddings
Representam palavras e textos em vetores densos capazes de capturar significado semântico e contexto.

### Diferenças

| TF‑IDF | Embeddings |
|---------|------------|
| Frequência | Contexto |
| Vetores esparsos | Vetores densos |
| Fácil interpretação | Melhor compreensão semântica |

### Aplicação na Aura AI

Em versões futuras da solução, embeddings poderão complementar o TF‑IDF para melhorar a detecção de intenção, sentimento e oportunidades comerciais.


# 6. Insights de Negócio

Com base nas análises realizadas, a Aura AI pode:

### Oportunidades
- Identificar interesses recorrentes dos clientes.
- Apoiar estratégias de upsell e cross-sell.
- Priorizar clientes com maior potencial comercial.

### Riscos
- Detectar sinais de insatisfação.
- Identificar possíveis casos de churn.
- Mapear reclamações recorrentes.

### Ações Recomendadas
1. Monitorar automaticamente termos críticos.
2. Criar alertas para possíveis riscos de churn.
3. Gerar relatórios periódicos para gestores comerciais.
4. Evoluir a solução utilizando embeddings e modelos mais avançados.

## Conclusão

A aplicação de NLP permitiu transformar dados textuais não estruturados em informações analíticas úteis para o negócio. A solução Aura AI demonstra como técnicas de Ciência de Dados podem apoiar decisões comerciais de forma escalável e eficiente.
